In [1]:
import arcpy
from arcpy.sa import *
import os
import re

# --- PARAMÈTRES ---
shapefile  = r"C:\Users\admin123\Desktop\viirs_data\arcgis\shape_file\Mine_Nunavut_Mollweide_Contour_ville.shp"
zone_field = "Nom"
dmsp_dir   = r"C:\Users\admin123\Desktop\viirs_data\dmsp_image\dmsp_mollweide"
out_base   = r"C:\Users\admin123\Desktop\viirs_data\arcgis\resultat"
data_base  = r"C:\Users\admin123\Desktop\viirs_data\arcgis\data_base"

# --- SETUP ---
arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True
arcpy.env.workspace = dmsp_dir

for d in [out_base, data_base]:
    if not os.path.exists(d):
        os.makedirs(d)

# --- LECTURE DES COORDONNÉES ET AREA DU SHAPEFILE ---
coord_dict = {}
area_dict  = {}

with arcpy.da.SearchCursor(shapefile, [zone_field, "Latitude", "Longitude", "AREA"]) as cur:
    for row in cur:
        coord_dict[row[0]] = (row[1], row[2])
        area_dict[row[0]]  = row[3]

print(f"Coordonnées et surfaces chargées pour {len(coord_dict)} zones : {list(coord_dict.keys())}")

# --- TRAITEMENT RASTER PAR RASTER ---
rasters = arcpy.ListRasters("*", "TIF")
if not rasters:
    raise Exception("Aucun raster .tif trouvé dans dmsp_dir.")

tables_creees = []
merged_table = None

for ras in rasters:
    match = re.search(r"(?<=F\d{2})\d{4}", ras)
    if not match:
        print(f"⚠️  Année non trouvée dans : {ras} -> ignoré")
        continue
    year = match.group(0)
    print(f"Traitement : {ras} (année {year})")

    # 1. Supprimer les valeurs négatives (-> NoData)
    raster_obj     = Raster(os.path.join(dmsp_dir, ras))
    raster_cleaned = SetNull(raster_obj < 0, raster_obj)

    # 2. Zonal statistics
    out_table = os.path.join(out_base, f"dmsp_{year}.dbf")
    ZonalStatisticsAsTable(
        in_zone_data    = shapefile,
        zone_field      = zone_field,
        in_value_raster = raster_cleaned,
        out_table       = out_table,
        ignore_nodata   = "DATA",
        statistics_type = "ALL"
    )

    # 3. Ajout du champ Annee
    existing_fields = [f.name for f in arcpy.ListFields(out_table)]
    if "Annee" not in existing_fields:
        arcpy.AddField_management(out_table, "Annee", "SHORT")
    with arcpy.da.UpdateCursor(out_table, ["Annee"]) as cur:
        for row in cur:
            row[0] = int(year)
            cur.updateRow(row)

    # 4. Ajout des champs Latitude, Longitude et AREA
    existing_fields = [f.name for f in arcpy.ListFields(out_table)]
    for field_name, field_type in [("Latitude",  "DOUBLE"),
                                    ("Longitude", "DOUBLE"),
                                    ("AREA",      "DOUBLE")]:
        if field_name not in existing_fields:
            arcpy.AddField_management(out_table, field_name, field_type)

    with arcpy.da.UpdateCursor(out_table, [zone_field, "Latitude", "Longitude", "AREA"]) as cur:
        for row in cur:
            nom = row[0]
            if nom in coord_dict:
                row[1] = coord_dict[nom][0]   # Latitude
                row[2] = coord_dict[nom][1]   # Longitude
                row[3] = area_dict[nom]        # AREA
                cur.updateRow(row)
            else:
                print(f"  ⚠️  Zone '{nom}' absente du shapefile, champs non remplis")

    tables_creees.append(out_table)
    print(f"  ✅ Table créée : {out_table}")

print("Zonal statistics terminé pour toutes les années trouvées.")

# --- FUSION ---
if tables_creees:
    merged_table = os.path.join(data_base, "dmsp_zonal_mine_2000_2013_moll.dbf")
    arcpy.Merge_management(tables_creees, merged_table)
    print(f"Table fusionnée créée : {merged_table}")
else:
    print("Aucune table créée, rien à fusionner.")
    merged_table = None
    
# --- EXPORT CSV ---
if merged_table and arcpy.Exists(merged_table):
    try:
        csv_name   = "dmsp_zonal_mine_2000_2013_moll.csv"
        csv_output = os.path.join(out_base, csv_name)
        arcpy.conversion.TableToTable(
            in_rows  = merged_table,
            out_path = out_base,
            out_name = csv_name
        )
        print(f"✅ Export CSV terminé : {csv_output}")
    except arcpy.ExecuteError:
        print("❌ Erreur ArcPy lors de l'export CSV")
        print(arcpy.GetMessages(2))
    except Exception as e:
        print(f"❌ Erreur lors de l'export CSV : {e}")
else:
    print("Aucune table fusionnée disponible, export CSV ignoré.")

# --- SUPPRESSION DES TABLES INTERMÉDIAIRES ---
for t in tables_creees:
    if arcpy.Exists(t):
        arcpy.Delete_management(t)
        print(f"🗑️  Supprimé : {t}")
print("✅ Toutes les tables intermédiaires ont été supprimées.")
print("✅ Script terminé.")

Coordonnées et surfaces chargées pour 11 zones : ['Mine_Meliadine', 'Mine_Baffinland', 'Mine_Hope_Bay', 'Mine_Meadowbank', 'Polaris_mine', 'Mine_Lupin', 'Jericho_Diamond_Mine', ' Ulu', 'Muskox_Project', 'Ferguson_lake', 'Storm_Copper']
Traitement : F152000.v4b.global.intercal.stable_lights.avg_vis_mollweide.tif (année 2000)
  ✅ Table créée : C:\Users\admin123\Desktop\viirs_data\arcgis\resultat\dmsp_2000.dbf
Traitement : F152001.v4b.global.intercal.stable_lights.avg_vis_mollweide.tif (année 2001)
  ✅ Table créée : C:\Users\admin123\Desktop\viirs_data\arcgis\resultat\dmsp_2001.dbf
Traitement : F152002.v4b.global.intercal.stable_lights.avg_vis_mollweide.tif (année 2002)
  ✅ Table créée : C:\Users\admin123\Desktop\viirs_data\arcgis\resultat\dmsp_2002.dbf
Traitement : F152003.v4b.global.intercal.stable_lights.avg_vis_mollweide.tif (année 2003)
  ✅ Table créée : C:\Users\admin123\Desktop\viirs_data\arcgis\resultat\dmsp_2003.dbf
Traitement : F162004.v4b.global.intercal.stable_lights.avg_vis_m